# Specialiserede Modeller — 4: Clustering — mønstre UDEN labels

De fleste ML-modeller trænes med en **facitliste**: giftig/spiselig, syg/rask,
hvilket ciffer. Men i den virkelige verden har data tit INGEN labels — kun rå
observationer. Kan en model overhovedet lære noget så? Ja: den kan finde **grupper**.
Det hedder **unsupervised learning**, og dagens algoritme er klassikeren **k-means**.

Scenariet: du er dataanalytiker for et butikscenter. Du har 200 kunder med årsindkomst
og en "spending score" (1-100: hvor meget shopper de?). Chefen spørger: **"hvilke
kundetyper har vi?"** — og der findes ingen facitliste. Værsgo.

> Denne notebook er selvkørende — du kan tage emnets notebooks i den rækkefølge, du vil. Der er med vilje flere opgaver, end du kan nå. Opgaver mærket **(find fejlen)** indeholder en bevidst fejl, som skal findes og rettes. Nederst ligger et par **ekstra opgaver**, hvis du får lyst til mere.
>
> Noget af det her er nyt og kan føles udfordrende i starten — og det er helt okay. Vi forklarer hvert skridt så klart og tydeligt, vi kan, og der er et hint til hver opgave, hvis du går i stå. Tag dig endelig god tid.

## Setup

In [ ]:
# Henter kunde-data fra GitHub (Plan B: upload Mall_Customers.csv via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/Mall_Customers.csv

!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from helpers import plot_kmeans_steps

np.random.seed(42)

df = pd.read_csv("Mall_Customers.csv")
df.head()

# 1: k-means — find klumperne

Lad os først SE på kunderne. Vi plotter årsindkomst mod spending score:

In [ ]:
X_raw = df[["Annual Income (k$)", "Spending Score (1-100)"]].values.astype("float32")

plt.figure(figsize=(7, 5))
plt.scatter(X_raw[:, 0], X_raw[:, 1], alpha=0.7)
plt.xlabel("årsindkomst (tusind $)")
plt.ylabel("spending score")
plt.title("200 kunder — kan I se grupperne?")
plt.show()

Kig godt — der ER tydelige klumper (de fleste ser 5). Men vi vil ikke sidde og tegne
cirkler i hånden på 200 kunder... og slet ikke på 2 millioner. Vi vil have en algoritme.

## Algoritmen bag k-means

**k-means** finder $k$ klynger sådan her:

1. Placér $k$ **centre** tilfældigt (fx oven på $k$ tilfældige datapunkter).
2. **Tildel**: hvert punkt hører til det nærmeste center.
3. **Flyt**: hvert center flytter til gennemsnittet (middelpunktet) af sine punkter.
4. Gentag 2-3, indtil intet flytter sig.

Ingen gradienter, ingen tabsfunktion at differentiere — kun afstande og gennemsnit.
Først standardiserer vi (afstande på tværs af enheder kræver fælles skala — Intro-ML-
lektionen igen!), og så kører vi algoritmen SYNLIGT, trin for trin:

In [ ]:
X = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)     # standardisering

k = 5
rng = np.random.default_rng(42)
centers = X[rng.choice(len(X), size=k, replace=False)].copy()   # trin 1: start på k tilfældige kunder

centers_history = []
assignments_history = []

for iteration in range(6):
    # trin 2: afstand fra hvert punkt til hvert center → vælg det nærmeste
    distances = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
    assignment = distances.argmin(axis=1)

    centers_history.append(centers.copy())
    assignments_history.append(assignment.copy())

    # trin 3: flyt hvert center til gennemsnittet af dets punkter
    for j in range(k):
        centers[j] = X[assignment == j].mean(axis=0)

plot_kmeans_steps(X, centers_history, assignments_history)

Følg de sorte kryds (centrene) fra plot til plot: de vandrer ind i hver sin klump og
falder til ro — algoritmen er typisk færdig på en håndfuld iterationer. (Linjen med
`np.linalg.norm` regner alle 200×5 afstande på én gang — det er numpy-broadcasting og
må godt føles som sort magi; pointen er trin 2's "vælg nærmeste center".)

## Det professionelle værktøj: sklearn

I sklearn er det samme mønster som træerne — nu uden `y`, for der ER ingen labels:

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
kmeans.fit(X)                                  # bemærk: KUN X — ingen y!

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=kmeans.labels_, cmap="tab10", alpha=0.8)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            marker="X", s=250, color="black", edgecolors="white")
plt.xlabel("indkomst (standardiseret)")
plt.ylabel("spending score (standardiseret)")
plt.title("sklearn's KMeans, k = 5")
plt.show()

## Fra klynger til KUNDETYPER

Tallene 0-4 betyder ingenting i sig selv — fortolkningen er VORES arbejde. Vi hænger
klyngenumrene på kunderne og regner gennemsnit pr. klynge:

In [ ]:
df["klynge"] = kmeans.labels_

profile = df.groupby("klynge")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)
profile["antal"] = df["klynge"].value_counts().sort_index()
profile

Prøv at give hver klynge et navn ud fra tallene — fx "velhavende storshoppere"
(høj indkomst + høj score) eller "forsigtige velhavere" (høj indkomst + lav score).
Det er præcis den slags segmenter, marketingafdelinger bygger kampagner på.

## Hvordan vælger man k? Albue-metoden

k-means svarer ALTID med præcis $k$ klynger — også hvis $k$ er helt skævt. Et
pejlemærke: mål den samlede afstand fra punkterne til deres centre (kaldes **inertia**
— sklearn regner den ud for os) for forskellige $k$, og se hvor kurven "knækker" som
en albue: dér holder ekstra klynger op med at hjælpe for alvor.

In [ ]:
inertias = []
k_values = range(1, 11)
for k_test in k_values:
    km = KMeans(n_clusters=k_test, n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.plot(k_values, inertias, "o-")
plt.xlabel("k (antal klynger)")
plt.ylabel("inertia (samlet afstand til centre)")
plt.title("Albue-metoden — find knækket")
plt.show()

### Opgaver

##### Opgave 1.1
Vi ser på, om k-means lander samme sted, uanset hvor den starter.

Trin-for-trin-algoritmen ovenfor startede på fem tilfældige kunder. Prøv at ændre starten med `rng = np.random.default_rng(0)`, og kør gerne et par forskellige tal.

Læg mærke til, om centrene ender de samme steder hver gang. Kig på cellen ovenfor, hvor de sorte kryds vandrede ind i klumperne.

Hint: Gradient descent kan finde forskellige dale alt efter, hvor den starter — kan k-means også ramme forskellige løsninger?

In [ ]:
rng = np.random.default_rng(0)      # ← prøv 0, 1, 7 ...
centers = X[rng.choice(len(X), size=5, replace=False)].copy()
centers_history, assignments_history = [], []
for iteration in range(6):
    distances = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
    assignment = distances.argmin(axis=1)
    centers_history.append(centers.copy())
    assignments_history.append(assignment.copy())
    for j in range(5):
        centers[j] = X[assignment == j].mean(axis=0)
plot_kmeans_steps(X, centers_history, assignments_history)

##### Opgave 1.2
Vi ser på, hvad der sker, når vi beder om for få eller for mange klynger.

Prøv at køre sklearn-KMeans med `k = 2`, derefter `3` og til sidst `8`, og se på plottet hver gang. Prøv at sætte ord på, hvad algoritmen gør ved kunderne, når k er for lille — og når k er for stor.

Kig på cellen med albue-metoden ovenfor, hvor vi netop ledte efter det rigtige k.

Hint: Når k er for lille, må flere naturlige klumper dele ét center — og når k er for stor, hvad sker der så med en klump, der egentlig hører sammen?

In [ ]:
k_test = 2       # ← prøv 2, 3, 8
km = KMeans(n_clusters=k_test, n_init=10, random_state=42)
km.fit(X)
plt.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap="tab10", alpha=0.8)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            marker="X", s=250, color="black", edgecolors="white")
plt.title(f"k = {k_test}")
plt.show()

##### Opgave 1.3
Vi bruger albue-metoden til at finde et godt antal klynger.

Prøv at udfylde albue-loopet, så det træner en model for hvert k og gemmer modellens inertia. Aflæs så, hvor kurven knækker som en albue for kundedataene.

Se om det passer med det antal klumper, du selv så i det allerførste scatter-plot øverst.

Hint: Modellen trænes med `km.fit(X)`, og tallet du vil gemme til listen, hedder `km.inertia_`.

In [ ]:
inertias = []
for k_test in range(1, 11):
    km = KMeans(n_clusters=k_test, n_init=10, random_state=42)
    ...                      # <-- træn modellen på X (samme kald som ovenfor: km.fit(...))
    inertias.append(...)     # <-- gem modellens inertia til listen: km.inertia_

plt.plot(range(1, 11), inertias, "o-")
plt.xlabel("k")
plt.ylabel("inertia")
plt.show()

##### Opgave 1.4
Vi ser på, hvorfor standardisering er så vigtig for k-means.

Forestil dig, at indkomsten stod i kroner i stedet for tusind dollars. Prøv at køre cellen og se på klyngerne. Se om du kan forklare, hvorfor k-means pludselig ignorerer spending score næsten helt.

Husk: afstande på tværs af enheder kræver en fælles skala — ellers kommer den store enhed til at bestemme det hele.

Hint: Når indkomsten står i kroner, bliver de tal enorme sammenlignet med spending score (1-100). Hvilken af de to akser kommer så til at dominere afstanden?

In [ ]:
X_kroner = X_raw.copy()
X_kroner[:, 0] = X_kroner[:, 0] * 7000        # tusind $ → kroner (ca.)

km = KMeans(n_clusters=5, n_init=10, random_state=42)
km.fit(X_kroner)

plt.scatter(X_kroner[:, 0], X_kroner[:, 1], c=km.labels_, cmap="tab10", alpha=0.8)
plt.xlabel("årsindkomst (kroner)")
plt.ylabel("spending score")
plt.title("k-means på u-standardiserede data")
plt.show()

##### Opgave 1.5
Nu oversætter vi de tørre klyngenumre til rigtige kundetyper.

Prøv at udfylde `groupby`-profilen, så den grupperer kunderne efter deres klynge, og giv derefter hver af de 5 klynger et navn (fx "velhavende storshoppere"). Overvej, hvilken klynge du ville sende luksus-reklamer til — og hvilken der skulle have rabatkuponer.

Kig på profil-cellen længere oppe, hvor vi regnede gennemsnit pr. klynge.

Hint: Hvad hedder den kolonne, vi lige har lagt ind i `df`? Det er den, `groupby` skal gruppere efter.

In [ ]:
df["klynge"] = kmeans.labels_
profile = df.groupby(...)[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)   # <-- gruppér efter kolonnen "klynge"
profile["antal"] = df["klynge"].value_counts().sort_index()
profile
# dine navne:
# klynge 0: ...
# klynge 1: ...

##### Opgave 1.6 (find fejlen)
Vi ser på, hvad der sker, hvis man kommer til at tage en forkert kolonne med i clusteringen.

En kammerat har taget "alle talkolonnerne" med — og klyngerne er blevet til mærkeligt meningsløse striber. Prøv at køre cellen, kig på plottet, og find den kolonne, der intet har at gøre i en afstandsberegning.

Den bevidste fejl er der stadig — det er den, du skal finde og rette (ved at fjerne den forkerte kolonne fra listen).

Hint: Kig på det første navn i kolonnelisten. Er `CustomerID` (kun et løbenummer 1, 2, 3, ...) overhovedet en egenskab, det giver mening at måle afstand på?

In [ ]:
X_error = df[["CustomerID", "Annual Income (k$)", "Spending Score (1-100)"]].values.astype("float32")
X_error = (X_error - X_error.mean(axis=0)) / X_error.std(axis=0)

km = KMeans(n_clusters=5, n_init=10, random_state=42)
km.fit(X_error)

plt.scatter(X_error[:, 1], X_error[:, 2], c=km.labels_, cmap="tab10", alpha=0.8)
plt.xlabel("indkomst (std)")
plt.ylabel("spending score (std)")
plt.title("hmm... klyngerne giver ikke mening")
plt.show()

##### Opgave 1.7
Vi tænker over et svært spørgsmål: hvordan ved man, om en clustering er "god", når der ingen facitliste findes?

I supervised learning kunne vi måle accuracy mod en facitliste. Her er der ingen. Så hvordan ved du overhovedet, om dine klynger er "rigtige"? Kan to forskellige opdelinger begge være "gode"?

Prøv at finde på mindst ét konkret tjek, man kunne lave.

Hint: Tænk på, hvad der kendetegner en god klynge rent geometrisk — punkter der ligger tæt på deres eget center og langt fra de andre centre. Kan det måles (kig evt. på `inertia` fra albue-metoden)?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.8
Vi laver en helt anden segmentering af de samme kunder.

Prøv at cluster på **Age** og **Spending Score** i stedet for indkomst (husk at standardisere først!). Se, hvilke nye grupper der dukker op, og overvej, hvilket k albue-metoden ville foreslå her.

Kig på setup-cellen øverst, hvor vi standardiserede med `(X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)`.

Husk: Uden standardisering kommer den akse med de største tal til at bestemme klyngerne (det så vi i opgave 1.4).

In [ ]:
X2_raw = df[["Age", "Spending Score (1-100)"]].values.astype("float32")
X2 = ...          # <-- standardisér X2_raw (samme formel som i setup: (data - mean) / std)

km2 = KMeans(n_clusters=4, n_init=10, random_state=42)
km2.fit(X2)
plt.scatter(X2[:, 0], X2[:, 1], c=km2.labels_, cmap="tab10", alpha=0.8)
plt.xlabel("alder (std)")
plt.ylabel("spending score (std)")
plt.show()

##### Opgave 1.9
Vi ser på k-means' akilleshæl: former, der ikke er runde klumper.

Prøv at køre k-means på måne-data (`make_moons`) og se på plottet. Overvej, hvorfor den fejler her, når et neuralt netværk godt kunne skille de to måner.

Kig på plottet med "de RIGTIGE grupper" ved siden af — de to måner slynger sig ind i hinanden.

Hint: "Nærmeste center" deler altid planen op med lige streger imellem centrene. Kan en lige streg nogensinde skille to måner, der krummer om hinanden?

In [ ]:
from sklearn.datasets import make_moons

X_m, y_correct = make_moons(n_samples=400, noise=0.05, random_state=42)

km_m = KMeans(n_clusters=2, n_init=10, random_state=42)
km_m.fit(X_m)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(X_m[:, 0], X_m[:, 1], c=y_correct, cmap="tab10", s=15)
axes[0].set_title("de RIGTIGE grupper")
axes[1].scatter(X_m[:, 0], X_m[:, 1], c=km_m.labels_, cmap="tab10", s=15)
axes[1].set_title("k-means' bud")
plt.show()

##### Opgave 1.12
Vi slutter af med at tænke over, hvad clustering kan bruges til — på godt og ondt.

Butikscentret vil bruge dine segmenter til målrettede tilbud og forskellige priser til forskellige kundetyper. Overvej: hvad kunne det bruges fornuftigt til — og hvor går den etiske grænse?

Tænk fx på klyngen "lav indkomst + høj spending score".

Hint: Er det rimeligt at vise netop de kunder, der shopper mest i forhold til deres indkomst, de dyreste priser? Hvem gavner det, og hvem rammer det?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

## Ekstra opgaver

Her er nogle ekstra udfordringer, hvis du er nået hele vejen igennem og har lyst til mere. De bygger videre på det, du allerede har lavet, og du kan tage dem i den rækkefølge, du vil.

##### Ekstra 1
Vi samler alt: en fuld segmentering på alle tre features.

Prøv at cluster på **Age**, **Annual Income** og **Spending Score** på én gang. Kør albue-metoden, vælg et k ud fra knækket, og lav klynge-profilen med `groupby`. Se om du stadig kan give hver klynge et sigende navn.

Skabelonen nedenfor følger de samme tre trin som i opgave 1.3 og 1.5 — den mangler kun, at du vælger dit k.

Hint: 3D kan ikke plottes på en flad skærm, så `groupby`-profilen ER dit billede af klyngerne — læs gennemsnittene som en beskrivelse af hver kundetype.

In [ ]:
X3_raw = df[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].values.astype("float32")
X3 = (X3_raw - X3_raw.mean(axis=0)) / X3_raw.std(axis=0)

# Trin 1: albue-metoden — find et godt k (samme loop som i opgave 1.3)
inertias = []
for k_test in range(1, 11):
    km = KMeans(n_clusters=k_test, n_init=10, random_state=42)
    km.fit(X3)
    inertias.append(km.inertia_)
plt.plot(range(1, 11), inertias, "o-")
plt.xlabel("k")
plt.ylabel("inertia")
plt.show()

# Trin 2: vælg dit k ud fra albuen og træn en model
k_valgt = ...                    # <-- vælg et tal ud fra albuen ovenfor, fx 5
km3 = KMeans(n_clusters=k_valgt, n_init=10, random_state=42)
km3.fit(X3)

# Trin 3: hæng klyngerne på kunderne og lav profilen (som i opgave 1.5)
df["klynge3"] = km3.labels_
profile3 = df.groupby("klynge3")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)
profile3["antal"] = df["klynge3"].value_counts().sort_index()
profile3

##### Ekstra 2
Vi kigger under motorhjelmen på tallet `inertia_`, som albue-metoden byggede på.

`inertia_` er summen af kvadrerede afstande fra hvert punkt til dets eget center. Prøv at udfylde loopet, så du regner den selv for k=5-modellen, og tjek at du rammer sklearn's tal.

Kig på albue-cellen ovenfor, hvor vi brugte `km.inertia_` uden at vide, hvad der gemte sig bag.

Hint: I numpy kan afstandsbidraget fra én klynge skrives på én linje: `((points - center)**2).sum()` — træk centret fra hvert punkt, kvadrér, og summér.

In [ ]:
km = KMeans(n_clusters=5, n_init=10, random_state=42)
km.fit(X)

total = 0.0
for j in range(5):
    points = X[km.labels_ == j]          # alle punkter i klynge j
    center = km.cluster_centers_[j]      # klynge j's center
    squared = ...                        # <-- for hvert punkt: træk center fra, kvadrér, og læg det hele sammen
    total = total + squared

print("vores:  ", round(total, 2))
print("sklearn:", round(km.inertia_, 2))